In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install owlready2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 44.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for owlready2: filename=owlready2-0.51-cp313-cp313-linux_x86_64.whl size=24764653 sha256=fac87446ceb8548445a1eb097a2781cdcd9370c55d7aa970b9940456c32d6cac
  Stored in directory: /root/.cache/pip/wheels/cc/16/52/762fdfa8c53f256db44674790a3ceba702c3c763c96ce8b62c
Successfully built owlready2


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import pandas as pd
import owlready2 as or2
from sklearn.preprocessing import StandardScaler

In [ ]:
onto = or2.get_ontology("/path/to/ontology/mlprov.owx").load()
with onto:
    class description(or2.DataProperty):
        range = [str]

* Owlready2 * WARNING: ObjectProperty http://www.w3.org/ns/prov#wasRevisionOf belongs to more than one entity types: [owl.AnnotationProperty, owl.ObjectProperty, prov.wasDerivedFrom]; I'm trying to fix it...
* Owlready2 * WARNING: ObjectProperty http://www.w3.org/ns/prov#specializationOf belongs to more than one entity types: [owl.AnnotationProperty, owl.ObjectProperty, prov.alternateOf]; I'm trying to fix it...


In [5]:
req_spec = onto["requirement_specification"]
train_data = onto["training_dataset"]
test_data = onto["testing_dataset"]
data_source = onto["data_source"]
method_collection = onto["Method_of_collection"]
data_collector = onto["data_collector"]
preprocessing_step = onto["preprocessing_step"]
exclusion_criteria = onto["exclusion_criteria"]
class_proportion = onto["dataset_class_split"]
onto_model = onto["model"]
performance_metric = onto["performance_metric"]

In [6]:
req1 = req_spec("req1")
req1.description = ["The model shall predict the class <=50K."]
req2 = req_spec("req2")
req2.description = ["The model shall predict the class >50K."]

training_dataset = train_data("adult_train.csv")
training_dataset.description = ["Location: https://github.com/AnonymousWriter1/BiasProvenance"]
original_test_dataset = test_data("adult_test.csv")
original_test_dataset.description = ["Location: https://github.com/AnonymousWriter1/BiasProvenance"]
sex_test_dataset = test_data("sex_test.csv")
sex_test_dataset.description = ["Location: https://github.com/AnonymousWriter1/BiasProvenance"]
race_test_dataset = test_data("race_test.csv")
race_test_dataset.description = ["Location: https://github.com/AnonymousWriter1/BiasProvenance"]

source = data_source("1994_Census")
method_of_collection = method_collection("Census_survey")
method_of_collection.description = ["Census survey conducted by the US Census Bureau. Year: 1994"]
collector = data_collector("Government_Official")

drop_na = preprocessing_step("Drop_NA")
excluded = exclusion_criteria("NA")
male = class_proportion("Percentage_Male=0.67")
female = class_proportion("Percentage_Female=0.33")
white = class_proportion("Percentage_White=0.85")
black = class_proportion("Percentage_Black=0.09")
api = class_proportion("Percentage_Asian_Pacific_Islander=0.03")
ai = class_proportion("Percentage_American_Indian=0.009")
other = class_proportion("Percentage_Other_Race=0.008")


In [ ]:
csv_file_path = '/path/to/data/adult_train.csv'
df = pd.read_csv(csv_file_path)
df = df.dropna()
df.head()


,Age,Workclass,fnlwgt,Education,Education_Num,Martial_Status,Occupation,Relationship,Race,Sex,Capital_Gain,Capital_Loss,Hours_per_week,Country,Target
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [8]:
X = df.iloc[:, :-1]   # all rows, all columns except the last
y = df.iloc[:, -1]    # all rows, just the last column

x_encoded = pd.get_dummies(X)
y_encoded = pd.get_dummies(y)
y_encoded = y.map({' <=50K': 0, ' >50K': 1})

In [9]:
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_encoded)

In [10]:
clf = LogisticRegression(max_iter=10000, random_state=0).fit(x_scaled, y_encoded)
#model = onto_model("clf")

In [11]:
y_pred = clf.predict(x_encoded)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


Test on original testing dataset

In [ ]:
df_test = pd.read_csv('/path/to/data/adult_test.csv')
df_test = df_test.dropna()
df_test.head()

x_test_unfiltered = df_test.iloc[:, :-1]
y_test_unfiltered = df_test.iloc[:, -1]
x_test_unfiltered_encoded = pd.get_dummies(x_test_unfiltered)
# Align the columns of the test set with the training set
x_test_unfiltered_encoded = x_test_unfiltered_encoded.reindex(columns=x_encoded.columns, fill_value=0)
y_test_unfiltered_encoded = pd.get_dummies(y_test_unfiltered)
y_test_unfiltered_encoded = y_test_unfiltered.map({' <=50K.': 0, ' >50K.': 1})

x_test_scaled = scaler.fit_transform(x_test_unfiltered_encoded)

y_pred_unfiltered = clf.predict(x_test_scaled)
acc_test_unfiltered = accuracy_score(y_test_unfiltered_encoded, y_pred_unfiltered) * 100
print(acc_test_unfiltered)
accuracy_test_unfiltered = performance_metric(f"Original_test_accuracy={acc_test_unfiltered}")

prec_test_unfiltered = precision_score(y_test_unfiltered_encoded, y_pred_unfiltered)
print(prec_test_unfiltered)
precision_test_unfiltered = performance_metric(f"Original_test_precision={prec_test_unfiltered}")

rec_test_unfiltered = recall_score(y_test_unfiltered_encoded, y_pred_unfiltered)
print(rec_test_unfiltered)
recall_test_unfiltered = performance_metric(f"Original_test_recall={rec_test_unfiltered}")

f1_test_unfiltered = f1_score(y_test_unfiltered_encoded, y_pred_unfiltered)
print(f1_test_unfiltered)
f1_score_test_unfiltered = performance_metric(f"Original_test_f1={f1_test_unfiltered}")

auc_test_unfiltered = roc_auc_score(y_test_unfiltered_encoded, y_pred_unfiltered)
print(auc_test_unfiltered)
auc_score_test_unfiltered = performance_metric(f"Original_test_auc={auc_test_unfiltered}")

84.6547144754316
0.741229593608892
0.5767567567567567
0.6487308101535187
0.7555878854206319


In [14]:
cm = confusion_matrix(y_test_unfiltered_encoded, y_pred_unfiltered)
print(cm)
tn, fp, fn, tp = cm.ravel()
print(tn, fp, fn, tp)

orig_tpr = tp / (tp + fn)
print(orig_tpr)
orginal_tpr = performance_metric(f"Original_test_TPR={orig_tpr}")

orig_fpr = fp / (fp + tn)
print(orig_fpr)
original_fpr = performance_metric(f"Original_test_FPR={orig_fpr}")


orig_tnr = tn / (tn + fp)
print(orig_tnr)
original_tnr = performance_metric(f"Original_test_TNR={orig_tnr}")


orig_fnr = fn / (fn + tp)
print(orig_fnr)
original_fnr = performance_metric(f"Original_test_FNR={orig_fnr}")

[[10615   745]
 [ 1566  2134]]
10615 745 1566 2134
0.5767567567567567
0.06558098591549295
0.934419014084507
0.42324324324324325


Minority sex test set

In [ ]:
df_test_filtered = pd.read_csv('/path/to/data/sex_test.csv')
df_test_filtered = df_test_filtered.dropna()
df_test_filtered.head()

x_test_filtered = df_test_filtered.iloc[:, :-1]
y_test_filtered = df_test_filtered.iloc[:, -1]
x_test_filtered_encoded = pd.get_dummies(x_test_filtered)
# Align the columns of the test set with the training set
x_test_filtered_encoded = x_test_filtered_encoded.reindex(columns=x_encoded.columns, fill_value=0)
y_test_filtered_encoded = pd.get_dummies(y_test_filtered)
y_test_filtered_encoded = y_test_filtered.map({' <=50K.': 0, ' >50K.': 1})

y_pred_filtered = clf.predict(x_test_filtered_encoded)
acc_test_sex = accuracy_score(y_test_filtered_encoded, y_pred_filtered) * 100
print(acc_test_sex)
accuracy_test_sex = performance_metric(f"Minority_sex_accuracy={acc_test_sex}")

prec_test_sex = precision_score(y_test_filtered_encoded, y_pred_filtered)
print(prec_test_sex)
precision_test_sex = performance_metric(f"Minority_sex_precision={prec_test_sex}")

rec_test_sex = recall_score(y_test_filtered_encoded, y_pred_filtered)
print(rec_test_sex)
recall_test_sex = performance_metric(f"Minority_sex_recall={rec_test_sex}")

f1_test_sex = f1_score(y_test_filtered_encoded, y_pred_filtered)
print(f1_test_sex)
f1_score_test_sex = performance_metric(f"Minority_sex_f1={f1_test_sex}")

auc_test_sex = roc_auc_score(y_test_filtered_encoded, y_pred_filtered)
print(auc_test_sex)
auc_score_test_sex = performance_metric(f"Minority_sex_auc={auc_test_sex}")

11.337268471402401
0.11337268471402402
1.0
0.2036563071297989
0.5


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


In [16]:
cm = confusion_matrix(y_test_filtered_encoded, y_pred_filtered)
print(cm)
tn, fp, fn, tp = cm.ravel()
print(tn, fp, fn, tp)


sex_tpr = tp / (tp + fn)
print(sex_tpr)
sex_test_tpr = performance_metric(f"Sex_test_TPR={sex_tpr}")

sex_fpr = fp / (fp + tn)
print(sex_fpr)
sex_test_fpr = performance_metric(f"Sex_test_FPR={sex_fpr}")

sex_tnr = tn / (tn + fp)
print(sex_tnr)
sex_test_tnr = performance_metric(f"Sex_test_TNR={sex_tnr}")

sex_fnr = fn / (fn + tp)
print(sex_fnr)
sex_test_fnr = performance_metric(f"Sex_test_FNR={sex_fnr}")

[[   0 4356]
 [   0  557]]
0 4356 0 557
1.0
1.0
0.0
0.0


Minority race test

In [ ]:
df_test_race = pd.read_csv('/path/to/data/race_test.csv')
df_test_race = df_test_race.dropna()
df_test_race.head()

x_test_race = df_test_race.iloc[:, :-1]
y_test_race = df_test_race.iloc[:, -1]
x_test_race_encoded = pd.get_dummies(x_test_race)
# Align the columns of the test set with the training set
x_test_race_encoded = x_test_race_encoded.reindex(columns=x_encoded.columns, fill_value=0)
y_test_race_encoded = pd.get_dummies(y_test_race)
y_test_race_encoded = y_test_race.map({' <=50K.': 0, ' >50K.': 1})

y_pred_race = clf.predict(x_test_race_encoded)
acc_test_race = accuracy_score(y_test_race_encoded, y_pred_race) * 100
print(acc_test_race)
accuracy_test_race = performance_metric(f"Minority_race_accuracy={acc_test_race}")

prec_test_race = precision_score(y_test_race_encoded, y_pred_race)
print(prec_test_race)
precision_test_race = performance_metric(f"Minority_race_precision={prec_test_race}")

rec_test_race = recall_score(y_test_race_encoded, y_pred_race)
print(rec_test_race)
recall_test_race = performance_metric(f"Minority_race_recall={rec_test_race}")

f1_test_race = f1_score(y_test_race_encoded, y_pred_race)
print(f1_test_sex)
f1_score_test_race = performance_metric(f"Minority_race_f1={f1_test_race}")

auc_test_race = roc_auc_score(y_test_race_encoded, y_pred_race)
print(auc_test_race)
auc_score_test_race = performance_metric(f"Minority_race_auc={auc_test_race}")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(


15.885167464114833
0.15885167464114833
1.0
0.2036563071297989
0.5


In [18]:
cm = confusion_matrix(y_test_race_encoded, y_pred_race)
print(cm)
tn, fp, fn, tp = cm.ravel()
print(tn, fp, fn, tp)


race_tpr = tp / (tp + fn)
print(race_tpr)
race_test_tpr = performance_metric(f"Minority_race_TPR={race_tpr}")

race_fpr = fp / (fp + tn)
print(race_fpr)
race_test_fpr = performance_metric(f"Minority_race_FPR={race_fpr}")

race_tnr = tn / (tn + fp)
print(race_tnr)
race_test_tnr = performance_metric(f"Minority_race_TNR={race_tnr}")


race_fnr = fn / (fn + tp)
print(race_fnr)
race_test_fnr = performance_metric(f"Minority_race_FNR={race_fnr}")

[[   0 1758]
 [   0  332]]
0 1758 0 332
1.0
1.0
0.0
0.0


Save provenance graph

In [ ]:
onto.save(file="/path/mlprov_feature_bias_instances.owl")